# Análise real das tentativas de calibração
Fonte: diagnóstico do usuário de 11/11/2026. Este notebook usa agregados sem vetores oculares. As quatro tentativas não representam quatro pessoas. Para recalcular a geometria original, execute scripts/analyze_calibration.py --diagnostic com seu arquivo privado. As métricas não avaliam aprendizado de máquina: descrevem conferências da interpolação geométrica.


In [1]:
from pathlib import Path
import json, math, sys
PROJECT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT / 'scripts'))
from analyze_calibration import charts
source = PROJECT / 'docs/reports/evidencias/analise-calibracao.json'
data = json.loads(source.read_text(encoding='utf-8'))
print('SHA-256 da fonte original:', data['source_sha256'])
print('Tentativas registradas:', len(data['attempts']))


SHA-256 da fonte original: 9099920c6f30e216dac8036a5ac330206d8a95fc56dc5492557d595419dd9056
Tentativas registradas: 4


In [2]:
for row in data['attempts']:
    if row['status'] != 'completed':
        print('Tentativa', row['attempt'], 'interrompida:', row['reason'])
        continue
    assert row['predicted'] + row['unsupported'] == row['total']
    # Reconstitui o RMSE geral por médias quadráticas ponderadas dos alvos.
    weighted = sum((t['count']-t['unsupported'])*t['rmse']**2 for t in row['targets'] if t['rmse'] is not None)
    rmse = math.sqrt(weighted/row['predicted'])
    assert math.isclose(rmse, row['rmse'], rel_tol=1e-9)
    print(f"Tentativa {row['attempt']}: RMSE {rmse:.6f}; sem previsão {100*row['unsupported']/row['total']:.1f}%; aprovada={row['approved']}")


Tentativa 1: RMSE 0.107122; sem previsão 49.0%; aprovada=False
Tentativa 2 interrompida: incomplete_geometry
Tentativa 3 interrompida: communication
Tentativa 4: RMSE 0.121876; sem previsão 23.5%; aprovada=False


In [3]:
figures = charts(data)
for name, drawing in figures:
    print(name, ': gráfico reproduzível de', drawing.width, 'x', drawing.height)
print('Regenerar SVG/PDF: python scripts/analyze_calibration.py --aggregate docs/reports/evidencias/analise-calibracao.json')


erro : gráfico reproduzível de 500 x 240
cobertura : gráfico reproduzível de 500 x 240
alvos-tentativa-1 : gráfico reproduzível de 500 x 360
alvos-tentativa-4 : gráfico reproduzível de 500 x 360
Regenerar SVG/PDF: python scripts/analyze_calibration.py --aggregate docs/reports/evidencias/analise-calibracao.json


## Análise crítica
Na tentativa 4 há menor ausência de previsão (23,5% versus 49,0%), mas maior RMSE nos quadros previstos (0,122 versus 0,107). Não é demonstração de melhora geral. Quadros não são amostras independentes de pessoas. Sem previsão não equivale a erro zero, ausência de olhar ou placa ignorada.

Ambas as tentativas concluídas foram reprovadas. Não há resultados oficiais de localização de placas, comparação A/B nem demonstração de precisão do tracking livre. A fonte não permite confirmar FPS, resolução ou condições objetivas de iluminação. Não há inferência de etnia, treinamento, classificação ou regressão próprios.
